In [468]:
import pandas as pd
import numpy as np


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse
import itertools
import random

In [469]:
book_tags = pd.read_csv('../data/book_tags.csv')
books = pd.read_csv('../data/books.csv')
ratings = pd.read_csv('../data/ratings.csv')
tags = pd.read_csv('../data/tags.csv')
to_read = pd.read_csv('../data/to_read.csv')

In [470]:
book_tags

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
...,...,...,...
999907,33288638,21303,7
999908,33288638,17271,7
999909,33288638,1126,7
999910,33288638,11478,7


In [471]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')

In [472]:
books

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,7130616,7130616,7392860,19,441019455,9.780441e+12,Ilona Andrews,2010.0,Bayou Moon,...,17204,18856,1180,105,575,3538,7860,6778,https://images.gr-assets.com/books/1307445460m...,https://images.gr-assets.com/books/1307445460s...
9996,9997,208324,208324,1084709,19,067973371X,9.780680e+12,Robert A. Caro,1990.0,Means of Ascent,...,12582,12952,395,303,551,1737,3389,6972,https://s.gr-assets.com/assets/nophoto/book/11...,https://s.gr-assets.com/assets/nophoto/book/50...
9997,9998,77431,77431,2393986,60,039330762X,9.780393e+12,Patrick O'Brian,1977.0,The Mauritius Command,...,9421,10733,374,11,111,1191,4240,5180,https://images.gr-assets.com/books/1455373531m...,https://images.gr-assets.com/books/1455373531s...
9998,9999,8565083,8565083,13433613,7,61711527,9.780062e+12,Peggy Orenstein,2011.0,Cinderella Ate My Daughter: Dispatches from th...,...,11279,11994,1988,275,1002,3765,4577,2375,https://images.gr-assets.com/books/1279214118m...,https://images.gr-assets.com/books/1279214118s...


In [473]:
ratings

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [474]:
tags

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-
...,...,...
34247,34247,Ｃhildrens
34248,34248,Ｆａｖｏｒｉｔｅｓ
34249,34249,Ｍａｎｇａ
34250,34250,ＳＥＲＩＥＳ


In [475]:
to_read

,user_id,book_id
0,1,112
1,1,235
2,1,533
3,1,1198
4,1,1874
...,...,...
912700,53424,4716
912701,53424,4844
912702,53424,5907
912703,53424,7569


In [476]:
tags

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-
...,...,...
34247,34247,Ｃhildrens
34248,34248,Ｆａｖｏｒｉｔｅｓ
34249,34249,Ｍａｎｇａ
34250,34250,ＳＥＲＩＥＳ


# Tag names filtering to only genres

In [477]:
book_tags_named = book_tags.merge(tags, on='tag_id', how='inner')
book_tags_named

,goodreads_book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult
...,...,...,...,...
999907,33288638,21303,7,neighbors
999908,33288638,17271,7,kindleunlimited
999909,33288638,1126,7,5-star-reads
999910,33288638,11478,7,fave-author


In [478]:
top_tag_names = book_tags_named.groupby('tag_name')['count'].sum().sort_values(ascending=False)
top_tag_names.head(20)

tag_name
to-read               140718761
currently-reading       7507958
favorites               4503173
fiction                 3688819
fantasy                 3548157
young-adult             1848306
classics                1756920
books-i-own             1317235
romance                 1231926
owned                   1224279
ya                       898334
mystery                  872282
non-fiction              857901
historical-fiction       815421
series                   782637
science-fiction          703866
sci-fi                   597325
paranormal               542559
kindle                   506882
contemporary             486001
Name: count, dtype: int64

In [479]:
synonym_map = {
    'ya': 'young-adult',
    'sci-fi': 'science-fiction',
    'sci-fi-fantasy': 'science-fiction',
    'nonfiction': 'non-fiction',
    'favourites': 'favorites',
    'classic': 'classics',
    'graphic-novel': 'graphic-novels',
    'audiobooks': 'audiobook',
    'children': 'childrens',
    'children-s': 'childrens',
    'children-s-books': 'childrens',
    'picture-books': 'childrens',
    'ebooks': 'ebook',
    'novel': 'novels',
    'adult-fiction': 'adult',
}

book_tags_named['tag_name_clean'] = book_tags_named['tag_name'].replace(synonym_map)

top_tag_names = book_tags_named.groupby('tag_name_clean')['count'].sum().sort_values(ascending=False)

In [480]:
collection_pattern = re.compile(r'#\d+\s*-\s*\d+|\bbox\s*set\b|\bboxset\b|\bomnibus\b|\bcollection\b|\bcomplete\s+series\b|books?\s+\d+\s*-\s*\d+', re.IGNORECASE)
books['is_collection'] = books['title'].str.contains(collection_pattern, na=False)
collection_ids = set(books[books['is_collection']]['id'])

In [481]:
# values in tag names to include

genre_whitelist = {
    'fiction','fantasy','young-adult','classics','romance','mystery','non-fiction',
    'historical-fiction','science-fiction','paranormal','contemporary','horror',
    'childrens','thriller','history','dystopian','humor','literature','dystopia',
    'crime','graphic-novels','memoir','biography','philosophy','poetry','manga',
    'suspense','new-adult','drama','plays','short-stories','middle-grade',
    'mythology','psychology','business','religion','christian','erotica',
    'self-help','politics','war','coming-of-age','action','comedy','family',
    'vampires','supernatural','magic','adventure','teen','school',
}

genre_only = book_tags_named[book_tags_named['tag_name_clean'].isin(genre_whitelist)]
genre_totals = genre_only.groupby('tag_name_clean')['count'].sum().sort_values(ascending=False)

# Join to get title name and aggregate all version

In [482]:
genre_with_titles = genre_only.merge(
    books[['book_id', 'title']], left_on='goodreads_book_id', right_on='book_id', how='inner'
)

# aggregate all editions/versions under the same title
title_genre_counts = (
    genre_with_titles.groupby(['title', 'tag_name_clean'])['count']
    .sum()
    .reset_index()
)

In [483]:
n = 9

top_n_genres = (
    title_genre_counts
    .sort_values(['title', 'count'], ascending=[True, False])
    .groupby('title')
    .head(n)
)

In [484]:
genre_combined = (
    top_n_genres
    .assign(token=top_n_genres['tag_name_clean'].str.replace('-', '_'))
    .groupby('title')['token']
    .apply(lambda toks: ' '.join(toks))
    .reset_index()
    .rename(columns={'token': 'genre_combined'})
)

In [485]:
genre_combined

,title,genre_combined
0,"Angels (Walsh Family, #3)",fiction romance contemporary humor drama family
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",short_stories horror comedy fiction humor lite...
2,#GIRLBOSS,non_fiction memoir business self_help biograph...
3,'Salem's Lot,horror vampires thriller fiction paranormal su...
4,"'Tis (Frank McCourt, #2)",non_fiction memoir biography fiction history c...
...,...,...
9959,واحة الغروب,fiction historical_fiction literature history ...
9960,يوتوبيا,dystopia fiction science_fiction literature fa...
9961,ڤيرتيجو,fiction crime literature thriller mystery dram...
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,romance manga graphic_novels young_adult conte...


# total average across all books with same title

In [486]:
title_rating_totals = books.groupby('title')[['ratings_1','ratings_2','ratings_3','ratings_4','ratings_5']].sum().reset_index()

title_rating_totals['total_ratings'] = title_rating_totals[['ratings_1','ratings_2','ratings_3','ratings_4','ratings_5']].sum(axis=1)
title_rating_totals['avg_rating'] = (
    title_rating_totals['ratings_1']*1 +
    title_rating_totals['ratings_2']*2 +
    title_rating_totals['ratings_3']*3 +
    title_rating_totals['ratings_4']*4 +
    title_rating_totals['ratings_5']*5
) / title_rating_totals['total_ratings']

# Buckets for books one for each decade based on most popular adaptation

In [487]:
def to_decade(year):
    if pd.isna(year):
        return 'unknown'
    year = int(year)
    decade_start = (year // 10) * 10
    return f'{decade_start}s'

books['decade'] = books['original_publication_year'].apply(to_decade)

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_year = books.loc[idx, ['title', 'original_publication_year', 'ratings_count']].reset_index(drop=True)
title_popular_year['decade'] = title_popular_year['original_publication_year'].apply(to_decade)

In [488]:
title_popular_year

,title,original_publication_year,ratings_count,decade
0,"Angels (Walsh Family, #3)",2002.0,25680,2000s
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",2013.0,7443,2010s
2,#GIRLBOSS,2014.0,40090,2010s
3,'Salem's Lot,1975.0,228680,1970s
4,"'Tis (Frank McCourt, #2)",1999.0,40726,1990s
...,...,...,...,...
9959,واحة الغروب,2006.0,10365,2000s
9960,يوتوبيا,2008.0,31669,2000s
9961,ڤيرتيجو,2007.0,17001,2000s
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,2007.0,11232,2000s


# Author most popular adaptation

In [489]:
books['author_token'] = (
    books['authors']
    .str.split(',').str[0]
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('.', '', regex=False)
)

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_author = books.loc[idx, ['title', 'author_token', 'ratings_count']].reset_index(drop=True)

In [490]:
title_popular_author

,title,author_token,ratings_count
0,"Angels (Walsh Family, #3)",Marian_Keyes,25680
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",حسن_الجندي,7443
2,#GIRLBOSS,Sophia_Amoruso,40090
3,'Salem's Lot,Stephen_King,228680
4,"'Tis (Frank McCourt, #2)",Frank_McCourt,40726
...,...,...,...
9959,واحة الغروب,بهاء_طاهر,10365
9960,يوتوبيا,أحمد_خالد_توفيق,31669
9961,ڤيرتيجو,أحمد_مراد,17001
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,Meca_Tanaka,11232


In [491]:
book_titles = books.set_index(books['id'])['title'].to_dict()
book_authors = books.set_index(books['id'])['authors'].to_dict()

# Language based on most popular adaptation

In [492]:
lang_synonym_map = {'en-US': 'eng', 'en-GB': 'eng', 'en-CA': 'eng', 'en': 'eng'}
books['language_code_clean'] = books['language_code'].replace(lang_synonym_map)
books['language_code_clean'] = books['language_code_clean'].fillna('unknown')

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_language = books.loc[idx, ['title', 'language_code_clean', 'ratings_count']].reset_index(drop=True)

# Most popular adaptaion combined

there are 33 duplicates, different versions of the same book, this just solves that by using most popular version

In [493]:
idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_edition = books.loc[idx, [
    'id', 'title', 'author_token', 'original_publication_year', 'language_code_clean', 'ratings_count'
]].reset_index(drop=True)
title_popular_edition['decade'] = title_popular_edition['original_publication_year'].apply(to_decade)

title_popular_edition

,id,title,author_token,original_publication_year,language_code_clean,ratings_count,decade
0,3998,"Angels (Walsh Family, #3)",Marian_Keyes,2002.0,eng,25680,2000s
1,9610,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",حسن_الجندي,2013.0,ara,7443,2010s
2,2855,#GIRLBOSS,Sophia_Amoruso,2014.0,eng,40090,2010s
3,349,'Salem's Lot,Stephen_King,1975.0,eng,228680,1970s
4,2252,"'Tis (Frank McCourt, #2)",Frank_McCourt,1999.0,eng,40726,1990s
...,...,...,...,...,...,...,...
9959,8247,واحة الغروب,بهاء_طاهر,2006.0,ara,10365,2000s
9960,2588,يوتوبيا,أحمد_خالد_توفيق,2008.0,ara,31669,2000s
9961,3538,ڤيرتيجو,أحمد_مراد,2007.0,ara,17001,2000s
9962,9321,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,Meca_Tanaka,2007.0,jpn,11232,2000s


In [494]:
genre_combined_full = genre_combined.merge(title_popular_edition, on='title', how='left')
genre_combined_full = genre_combined_full.set_index('id')

In [495]:
genre_combined_full

,title,genre_combined,author_token,original_publication_year,language_code_clean,ratings_count,decade
id,,,,,,,
3998,"Angels (Walsh Family, #3)",fiction romance contemporary humor drama family,Marian_Keyes,2002.0,eng,25680,2000s
9610,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",short_stories horror comedy fiction humor lite...,حسن_الجندي,2013.0,ara,7443,2010s
2855,#GIRLBOSS,non_fiction memoir business self_help biograph...,Sophia_Amoruso,2014.0,eng,40090,2010s
349,'Salem's Lot,horror vampires thriller fiction paranormal su...,Stephen_King,1975.0,eng,228680,1970s
2252,"'Tis (Frank McCourt, #2)",non_fiction memoir biography fiction history c...,Frank_McCourt,1999.0,eng,40726,1990s
...,...,...,...,...,...,...,...
8247,واحة الغروب,fiction historical_fiction literature history ...,بهاء_طاهر,2006.0,ara,10365,2000s
2588,يوتوبيا,dystopia fiction science_fiction literature fa...,أحمد_خالد_توفيق,2008.0,ara,31669,2000s
3538,ڤيرتيجو,fiction crime literature thriller mystery dram...,أحمد_مراد,2007.0,ara,17001,2000s


# Splitting rating to 70/30 split ratio for ratings

In [496]:
id_to_title = books.set_index('id')['title'].to_dict()
title_to_canonical_id = title_popular_edition.set_index('title')['id'].to_dict()

ratings['title'] = ratings['book_id'].map(id_to_title)
ratings['canonical_id'] = ratings['title'].map(title_to_canonical_id)

ratings

,book_id,user_id,rating,title,canonical_id
0,1,314,5,"The Hunger Games (The Hunger Games, #1)",1
1,1,439,3,"The Hunger Games (The Hunger Games, #1)",1
2,1,588,5,"The Hunger Games (The Hunger Games, #1)",1
3,1,1169,4,"The Hunger Games (The Hunger Games, #1)",1
4,1,1185,4,"The Hunger Games (The Hunger Games, #1)",1
...,...,...,...,...,...
981751,10000,48386,5,The First World War,10000
981752,10000,49007,4,The First World War,10000
981753,10000,49383,5,The First World War,10000
981754,10000,50124,5,The First World War,10000


In [497]:
rating_counts = ratings['user_id'].value_counts()
eligible_users = rating_counts[rating_counts >= 10].index
ratings_eligible = ratings[ratings['user_id'].isin(eligible_users)].copy()

def split_group_three_way(group, train_frac=0.6, val_frac=0.2, seed=42):
    train = group.sample(frac=train_frac, random_state=seed)
    remaining = group.drop(train.index)
    val = remaining.sample(frac=val_frac / (1 - train_frac), random_state=seed)
    test = remaining.drop(val.index)
    return train, val, test

train_parts = []
val_parts = []
test_parts = []

for user_id, group in ratings_eligible.groupby('user_id'):
    train, val, test = split_group_three_way(group)
    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

ratings_train = pd.concat(train_parts).reset_index(drop=True)
ratings_val = pd.concat(val_parts).reset_index(drop=True)
ratings_test = pd.concat(test_parts).reset_index(drop=True)

In [498]:
ratings_train

,book_id,user_id,rating,title,canonical_id
0,1199,7,4,"Y: The Last Man, Vol. 1: Unmanned",1199
1,3246,7,4,"The Rise of Endymion (Hyperion Cantos, #4)",3246
2,1646,7,3,"Shadow of the Hegemon (Ender's Shadow, #2)",1646
3,585,7,4,"Death Note, Vol. 1: Boredom (Death Note, #1)",585
4,4459,7,4,The Elf Queen of Shannara (Heritage of Shannar...,4459
...,...,...,...,...,...
514848,8609,53424,4,The Lady of Shalott,8609
514849,7833,53424,4,"A Girl of the Limberlost (Limberlost, #2)",7833
514850,8213,53424,4,The Tale of Three Trees,8213
514851,4214,53424,5,A Single Shard,4214


In [499]:
ratings_test

,book_id,user_id,rating,title,canonical_id
0,956,7,5,"Surely You're Joking, Mr. Feynman!: Adventures...",956
1,1801,7,5,The Sword of Shannara (The Original Shannara T...,1801
2,1923,7,4,Barrel Fever: Stories and Essays,1923
3,2189,7,3,"The Ghost Brigades (Old Man's War, #2)",2189
4,2325,7,4,"The Player of Games (Culture, #2)",2325
...,...,...,...,...,...
171970,5811,53422,4,"Chosen at Nightfall (Shadow Falls, #5)",5811
171971,8757,53422,5,The Complete Works of H.P. Lovecraft,8757
171972,7212,53424,4,Captains Courageous,7212
171973,7503,53424,4,"What Katy Did (Carr Family, #1)",7503


# Feature Matrix

In [500]:
tfidf = TfidfVectorizer(token_pattern=r"[^\s]+")
genre_matrix = tfidf.fit_transform(genre_combined_full['genre_combined'])

ohe_author = OneHotEncoder(handle_unknown='ignore')
author_matrix = ohe_author.fit_transform(genre_combined_full[['author_token']].fillna('unknown'))

ohe_decade = OneHotEncoder(handle_unknown='ignore')
decade_matrix = ohe_decade.fit_transform(genre_combined_full[['decade']].fillna('unknown'))

ohe_lang = OneHotEncoder(handle_unknown='ignore')
lang_matrix = ohe_lang.fit_transform(genre_combined_full[['language_code_clean']].fillna('unknown'))

W_GENRE = 1
W_AUTHOR = 2
W_DECADE = 0.3
W_LANGUAGE = 0.3

feature_matrix = sparse.hstack([
    genre_matrix * W_GENRE,
    author_matrix * W_AUTHOR,
    decade_matrix * W_DECADE,
    lang_matrix * W_LANGUAGE,
]).tocsr()

id_to_row = {bid: i for i, bid in enumerate(genre_combined_full.index)}

In [501]:
def build_user_profile(user_id, like_threshold=4):

    # check for a particular user and if they rated a book above 4 stars
    user_ratings = ratings_train[ratings_train['user_id'] == user_id]
    liked = user_ratings[user_ratings['rating'] >= like_threshold]

    rows = []
    weights = []

    # find the id of the most popular verison of the book
    for _, r in liked.iterrows():
        row = id_to_row.get(r['canonical_id'])
        if row is not None:
            rows.append(row)
            weights.append(r['rating'])

    # cold start case
    if not rows:
        return None

    # give weighted average for the star ratings
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()
    liked_vecs = feature_matrix[rows]

    # gives user taste profile    
    profile = sparse.csr_matrix(weights) @ liked_vecs
    return profile, set(liked['canonical_id'])

def recommend_likes_only(user_id, n=10, like_threshold=4):
    result = build_user_profile(user_id, like_threshold)
    if result is None:
        return pd.DataFrame(columns=['canonical_id', 'title', 'authors', 'score'])
    profile, already_liked_ids = result

    train_rated_ids = set(ratings_train[ratings_train['user_id'] == user_id]['canonical_id'])  

    sims = cosine_similarity(profile, feature_matrix).flatten()
    id_list = list(id_to_row.keys())
    scores = pd.DataFrame({'canonical_id': id_list, 'score': sims})
    scores = scores[~scores['canonical_id'].isin(train_rated_ids)]
    scores = scores[~scores['canonical_id'].isin(collection_ids)]

    scores = scores.sort_values('score', ascending=False).head(n)
    scores['title'] = scores['canonical_id'].map(book_titles)
    scores['authors'] = scores['canonical_id'].map(book_authors)

    return scores[['canonical_id', 'title', 'authors', 'score']].reset_index(drop=True)

In [502]:
sample_user = ratings_train['user_id'].value_counts().index[0]  
recommendations = recommend_likes_only(sample_user, n=10)

recommendations.sort_values('score')
recommendations['title'].tolist()

recommendations

print(f"test size for this user is {len(ratings_test[ratings_test['user_id'] == sample_user])}")

ratings_test[ratings_test['user_id'] == sample_user].head(20)

test size for this user is 40


,book_id,user_id,rating,title,canonical_id
45498,7,12874,4,The Hobbit,7
45499,11,12874,5,The Kite Runner,11
45500,13,12874,4,1984,13
45501,29,12874,3,Romeo and Juliet,29
45502,42,12874,4,"Little Women (Little Women, #1)",42
45503,71,12874,2,Frankenstein,71
45504,83,12874,3,A Tale of Two Cities,83
45505,101,12874,4,Me Talk Pretty One Day,101
45506,113,12874,3,Catch-22,113
45507,118,12874,4,The Joy Luck Club,118


In [503]:
user_to_read_ids = set(ratings_test[ratings_test['user_id'] == sample_user]['canonical_id'])

is_hit = recommendations['canonical_id'].isin(user_to_read_ids)
hits = is_hit.sum()

print(f"{hits} / {len(recommendations)} recommendations are in this user's to-read list")

hit_rows = recommendations[is_hit]
hit_rows

1 / 10 recommendations are in this user's to-read list


,canonical_id,title,authors,score
3,2315,The Trumpet of the Swan,"E.B. White, Fred Marcellino",0.405237


In [504]:
ratings_train[ratings_train['user_id'] == sample_user]

,book_id,user_id,rating,title,canonical_id
136236,321,12874,4,Where the Red Fern Grows,321
136237,24,12874,4,Harry Potter and the Goblet of Fire (Harry Pot...,24
136238,59,12874,4,Charlotte's Web,59
136239,984,12874,3,Paradise Lost,984
136240,579,12874,3,Are You My Mother?,579
...,...,...,...,...,...
136351,544,12874,4,"Little House on the Prairie (Little House, #2)",544
136352,157,12874,4,Green Eggs and Ham,157
136353,102,12874,2,Where the Wild Things Are,102
136354,1167,12874,2,The Divine Comedy,1167


# Take a random subset of training as validation to tune for these hyper param

In [505]:
weight_grid = {
    'W_GENRE':    [1.0, 2.0, 3.0],
    'W_AUTHOR':   [2.0, 3.0, 4.0],
    'W_DECADE':   [0.0, 0.3, 0.5],
    'W_LANGUAGE': [0.0, 0.3, 0.5],
}

random.seed(42)
eval_sample = random.sample(list(ratings_train['user_id'].unique()), 500)  # smaller sample for speed during search

results = []

for w_genre, w_author, w_decade, w_lang in itertools.product(
    weight_grid['W_GENRE'], weight_grid['W_AUTHOR'], weight_grid['W_DECADE'], weight_grid['W_LANGUAGE']
):
    feature_matrix = sparse.hstack([
        genre_matrix * w_genre,
        author_matrix * w_author,
        decade_matrix * w_decade,
        lang_matrix * w_lang,
    ]).tocsr()

    id_to_row = {bid: i for i, bid in enumerate(genre_combined_full.index)}

    hit_rates = []
    for user_id in eval_sample:
        recommendations = recommend_likes_only(user_id, n=10)  # uses feature_matrix/id_to_row from closure
        if recommendations.empty:
            continue
        user_val_ids = set(ratings_val[ratings_val['user_id'] == user_id]['canonical_id'])
        if not user_val_ids:
            continue
        hits = recommendations['canonical_id'].isin(user_val_ids).sum()
        hit_rates.append(hits / len(recommendations))

    avg_hit_rate = sum(hit_rates) / len(hit_rates) if hit_rates else 0
    results.append({'W_GENRE': w_genre, 'W_AUTHOR': w_author, 'W_DECADE': w_decade, 'W_LANGUAGE': w_lang, 'avg_hit_rate': avg_hit_rate})

results_df = pd.DataFrame(results).sort_values('avg_hit_rate', ascending=False)
results_df.head(10)

,W_GENRE,W_AUTHOR,W_DECADE,W_LANGUAGE,avg_hit_rate
7,1.0,2.0,0.5,0.3,0.075605
16,1.0,3.0,0.5,0.3,0.075000
6,1.0,2.0,0.5,0.0,0.075000
15,1.0,3.0,0.5,0.0,0.074798
44,2.0,3.0,0.5,0.5,0.074798
42,2.0,3.0,0.5,0.0,0.074798
43,2.0,3.0,0.5,0.3,0.074798
39,2.0,3.0,0.3,0.0,0.074597
3,1.0,2.0,0.3,0.0,0.074395
53,2.0,4.0,0.5,0.5,0.074395


In [506]:
best_row = results_df.sort_values('avg_hit_rate', ascending=False).iloc[0]

W_GENRE = best_row['W_GENRE']
W_AUTHOR = best_row['W_AUTHOR']
W_DECADE = best_row['W_DECADE']
W_LANGUAGE = best_row['W_LANGUAGE'] 

# Eval only a subset of test since the size is too big

In [ ]:
eval_users = ratings_train['user_id'].unique()
k = 5


precisions= []
recalls = []
f1s = []
reciprocal_ranks = []


for user_id in eval_users:
    recommendations = recommend_likes_only(user_id, n=k)
    if recommendations.empty:
        continue

    user_test_ids = set(ratings_test[ratings_test['user_id'] == user_id]['canonical_id'])
    if not user_test_ids:
        continue

    is_hit = recommendations['canonical_id'].isin(user_test_ids)
    hits = is_hit.sum()

    precision_at_k = hits / len(recommendations)
    recall_at_k = hits / len(user_test_ids)

    if (precision_at_k + recall_at_k) > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    if hits > 0:
        first_hit_rank = is_hit.values.argmax() + 1
        reciprocal_rank = 1 / first_hit_rank
    else:
        reciprocal_rank = 0.0

    precisions.append(precision_at_k)
    recalls.append(recall_at_k)
    f1s.append(f1_at_k)
    reciprocal_ranks.append(reciprocal_rank)

avg_precision_at_k = sum(precisions) / len(precisions)
avg_recall_at_k = sum(recalls) / len(recalls)
avg_f1_at_k = sum(f1s) / len(f1s)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

print(f"evaluated {len(precisions)} users")
print(f"Precision@{k}: {avg_precision_at_k:.2%}")
print(f"Recall@{k}: {avg_recall_at_k:.2%}")
print(f"F1@{k}: {avg_f1_at_k:.2%}")
print(f"MRR: {mrr:.4f}")

evaluated 24264 users
Precision@5: 8.84%
Recall@5: 7.93%
F1@5: 7.49%
MRR: 0.1855


In [511]:
eval_users = ratings_train['user_id'].unique()
k = 10


precisions= []
recalls = []
f1s = []
reciprocal_ranks = []


for user_id in eval_users:
    recommendations = recommend_likes_only(user_id, n=k)
    if recommendations.empty:
        continue

    user_test_ids = set(ratings_test[ratings_test['user_id'] == user_id]['canonical_id'])
    if not user_test_ids:
        continue

    is_hit = recommendations['canonical_id'].isin(user_test_ids)
    hits = is_hit.sum()

    precision_at_k = hits / len(recommendations)
    recall_at_k = hits / len(user_test_ids)

    if (precision_at_k + recall_at_k) > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    if hits > 0:
        first_hit_rank = is_hit.values.argmax() + 1
        reciprocal_rank = 1 / first_hit_rank
    else:
        reciprocal_rank = 0.0

    precisions.append(precision_at_k)
    recalls.append(recall_at_k)
    f1s.append(f1_at_k)
    reciprocal_ranks.append(reciprocal_rank)

avg_precision_at_k = sum(precisions) / len(precisions)
avg_recall_at_k = sum(recalls) / len(recalls)
avg_f1_at_k = sum(f1s) / len(f1s)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

print(f"evaluated {len(precisions)} users")
print(f"Precision@{k}: {avg_precision_at_k:.2%}")
print(f"Recall@{k}: {avg_recall_at_k:.2%}")
print(f"F1@{k}: {avg_f1_at_k:.2%}")
print(f"MRR: {mrr:.4f}")


evaluated 24264 users
Precision@10: 7.02%
Recall@10: 11.94%
F1@10: 7.92%
MRR: 0.1994


In [512]:
eval_users = ratings_train['user_id'].unique()
k = 15


precisions= []
recalls = []
f1s = []
reciprocal_ranks = []


for user_id in eval_users:
    recommendations = recommend_likes_only(user_id, n=k)
    if recommendations.empty:
        continue

    user_test_ids = set(ratings_test[ratings_test['user_id'] == user_id]['canonical_id'])
    if not user_test_ids:
        continue

    is_hit = recommendations['canonical_id'].isin(user_test_ids)
    hits = is_hit.sum()

    precision_at_k = hits / len(recommendations)
    recall_at_k = hits / len(user_test_ids)

    if (precision_at_k + recall_at_k) > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    if hits > 0:
        first_hit_rank = is_hit.values.argmax() + 1
        reciprocal_rank = 1 / first_hit_rank
    else:
        reciprocal_rank = 0.0

    precisions.append(precision_at_k)
    recalls.append(recall_at_k)
    f1s.append(f1_at_k)
    reciprocal_ranks.append(reciprocal_rank)

avg_precision_at_k = sum(precisions) / len(precisions)
avg_recall_at_k = sum(recalls) / len(recalls)
avg_f1_at_k = sum(f1s) / len(f1s)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

print(f"evaluated {len(precisions)} users")
print(f"Precision@{k}: {avg_precision_at_k:.2%}")
print(f"Recall@{k}: {avg_recall_at_k:.2%}")
print(f"F1@{k}: {avg_f1_at_k:.2%}")
print(f"MRR: {mrr:.4f}")


evaluated 24264 users
Precision@15: 5.96%
Recall@15: 14.72%
F1@15: 7.66%
MRR: 0.2042
